# featherweight-ai — Kaggle QLoRA training run (Week 3)

Produces **benchmark row 2**. A thin launcher, not a place logic lives
(`docs/plan.md` §4): every cell below either configures the session or calls into
the repo.

**Before running — right sidebar:**

| Setting | Value |
|---|---|
| Accelerator | **GPU T4 x2** |
| Internet | **On** |
| Add-ons → Secrets | `HF_TOKEN` = your Hugging Face read token |

The model is gated (`gated: auto`) — a token alone is not enough, the licence must
also have been accepted on the model page. Those are two separate things and the
failure mode is a 403 that looks like a bad token (`memory.md` §8).

**Budget:** one run is **60 minutes of wall-clock training**, not a fixed number of
steps. `plan.md` §6 compares rows at equal wall-clock and equal peak VRAM — DoRA
costs more per step than LoRA, so matching on steps would quietly hand the slower
method more compute. Everything after the training cell is minutes, not hours.

## 1. What hardware did we actually get?

Record it. A wall-clock budget is only comparable across runs on the same device.

In [ ]:
!nvidia-smi

## 2. Clone the repo

Code lives in git; the notebook is a launcher. `rm -rf` first so a re-run inside the
same session picks up a fresh push rather than silently testing stale code.

In [ ]:
!rm -rf /kaggle/working/featherweight-ai
!git clone -q https://github.com/AadiPathak23/featherweight-ai.git /kaggle/working/featherweight-ai
%cd /kaggle/working/featherweight-ai
!git log --oneline -1

## 3. Install — the one genuinely untested piece

`requirements-kaggle.txt` has never run against this image. It deliberately does
**not** list `torch`: Kaggle's is preinstalled and matched to CUDA 12.8, and letting
pip resolve its own would mean a ~2.5 GB download and a real chance of a CUDA
mismatch that only fails at the first `.cuda()` call.

So the next cell **prints every resolved version**. If this run is ever compared
against a later one, the difference has to be visible here rather than inferred.

In [ ]:
!pip install -q -r requirements-kaggle.txt

In [ ]:
import torch, transformers, peft, bitsandbytes, accelerate, datasets, huggingface_hub, PIL, sys

print("python           :", sys.version.split()[0])
print("torch            :", torch.__version__, "| CUDA:", torch.version.cuda)
print("transformers     :", transformers.__version__, " (must be >=4.57, <5)")
print("peft             :", peft.__version__)
print("bitsandbytes     :", bitsandbytes.__version__)
print("accelerate       :", accelerate.__version__)
print("datasets         :", datasets.__version__)
print("huggingface_hub  :", huggingface_hub.__version__, " (must be <1.0)")
print("pillow           :", PIL.__version__)

print("\n--- devices ---")
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  cuda:{i} = {p.name}, capability=({p.major}, {p.minor}), {p.total_memory/1024**3:.2f} GiB")

# The constraint the whole project is built around. Bare is_bf16_supported() returns
# True on a T4 because PyTorch emulates bf16 through fp32 — functional, slow, none of
# the benefit. `including_emulation=False` is the truth. memory.md §5.
print("\nbf16 (bare, LIES on a T4) :", torch.cuda.is_bf16_supported())
try:
    print("bf16 (native, the truth)  :", torch.cuda.is_bf16_supported(including_emulation=False))
except TypeError:
    print("bf16 (native, the truth)  : <arg not in this torch>", torch.cuda.get_device_capability(0) >= (8, 0))

## 4. HF token from Secrets

Never typed into a cell. Kaggle notebooks are public by default and a committed
token is a leaked token — that has already happened once in this project
(`memory.md` §1 security note). Prints the length only.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
print("HF_TOKEN retrieved OK, length:", len(os.environ["HF_TOKEN"]))

## 5. Build the data

Two artifacts, both from committed manifests so the identity is fixed and only the
pixels are rebuilt:

- **eval split** — `day-validation` shards 0–7, 1,117 rows, frozen in Milestone E
- **training pool** — `day-train` shards 0–3, ~560 rows

`build_train_pool.py` runs a **hard scene-leak check** between them and exits
non-zero if any scene appears in both. nuScenes splits by scene so this should be
empty — but a leak would inflate row 2 and be completely invisible in the accuracy
number, which is why it is a failure and not a warning.

In [ ]:
!python scripts/build_eval_split.py

In [ ]:
!python scripts/build_train_pool.py

## 6. Measure the VRAM/batch-size curve *before* committing an hour to it

One point cannot separate the fixed cost (4-bit weights, the fp32 upcast, optimizer
state) from the part that scales with batch size (retained activations). Two can.
Costs about a minute and decides the `--batch-size` on the next cell.

In [ ]:
!python -m src.train --probe-batch --probe-max 32 --out train_batch_probe_t4.json

## 7. Train

`src/train.py` imports the Milestone F tripwire unchanged. §10 measured what it
catches: exactly one optimizer step lands, every step after it is silently skipped,
the loss stays finite and oscillating, and the progress bar keeps advancing. Here
that would be 12 hours and a saved adapter of pure garbage.

**Set `--batch-size` from the probe above.** The adapter is saved every 200 steps,
so a session that dies still leaves something scoreable.

In [ ]:
!python -m src.train \
    --budget-minutes 60 \
    --batch-size 1 \
    --grad-accum 8 \
    --lr 1e-4 \
    --save-every 200 \
    --run-name qlora \
    --out train_qlora.json

## 8. Score it — benchmark row 2

`src/eval.py`, **unmodified**, with `--adapter`. A row scored by different code is
not a row. Compare against row 1: **35.1% strict, 26.3% majority baseline,
+8.8 pp** (`memory.md` §9).

~6 minutes for 1,117 rows.

In [ ]:
!python -m src.eval --adapter outputs/adapters/qlora --run-name eval_qlora --out eval_qlora.json

## 9. Is the difference real?

Paired McNemar on the same 1,117 examples. Overlapping confidence intervals are the
wrong test here — only the discordant pairs carry information about which method is
better, and the paired test is what makes n=1,117 enough to rank methods a couple of
points apart.

In [ ]:
!python -m src.eval --compare results/eval_zeroshot.json results/eval_qlora.json

## 10. Get the artifacts out

`/kaggle/working` is wiped on teardown beyond what the session saves. Copy the
adapter and the results files out **before** stopping the session, then commit the
JSON back to the repo (`results/` is tracked; adapters are not — they belong on the
Hub).

In [ ]:
!mkdir -p /kaggle/working/artifacts
!cp -r outputs/adapters/qlora /kaggle/working/artifacts/
!cp results/train_qlora.json results/eval_qlora.json results/train_batch_probe_t4.json /kaggle/working/artifacts/
!cp outputs/adapters/qlora/train_log.jsonl /kaggle/working/artifacts/
!du -sh /kaggle/working/artifacts/* && ls -la /kaggle/working/artifacts

## 11. Success criteria

- [ ] Every version printed in §3; `transformers` 4.5x, `huggingface_hub` <1.0
- [ ] `bf16_native = False` — still true, still the reason this is an fp16 run
- [ ] Leak check reports **0 shared scenes**
- [ ] Training ends with `stop_reason = wall-clock budget exhausted`, **not** `tripwire`
- [ ] Scaler skips are a handful, not sustained (§10 baseline had 6 in 50 steps)
- [ ] `eval_qlora.json` strict accuracy recorded against row 1's **35.1%**
- [ ] Artifacts copied out **before** stopping the session

**Then stop the session explicitly.** An idle session burns the ~30 hr/week quota.